The `yolo11.yaml` configuration file has been modified to optimize the YOLO11n model for detecting shrimp fry. The changes include:

- Setting `nc` (number of classes) to 1, specifically for shrimp fry detection.
- Adjusting the `n` scale parameters to `[0.67, 0.33, 1024]` to potentially improve small object detection.
- Increasing the `repeats` parameter in all `C3k2` modules within both the `backbone` and `head` sections from 2 to 3, enhancing the model's capacity to learn more complex features.

**Reasoning for modifications:**

1.  **`nc: 1` (number of classes):** The original model was configured for 80 classes. By setting `nc` to 1, the model is specialized to detect only "shrimp fry," reducing complexity and improving focus for this specific task.

2.  **`n: [0.67, 0.33, 1024]` (scale parameters for depth and width):**
    *   **Depth (`0.67` from `0.50`):** A deeper network can learn more intricate representations, which is beneficial for detecting small objects with subtle features like shrimp fry.
    *   **Width (`0.33` from `0.25`):** More channels allow the network to process more information at each layer, helping to capture fine-grained details of small objects.

3.  **Increasing `repeats` in `C3k2` modules (from 2 to 3):** `C3k2` modules are crucial for feature extraction. Increasing their repeats makes these blocks deeper and more powerful, allowing the model to learn more robust and discriminative features, which is essential for accurate identification and localization of small objects.

In [3]:
from ultralytics import YOLO
import torch

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda:0") # Use the first GPU
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Training on CPU.")

Training on GPU: NVIDIA GeForce RTX 2060 SUPER


In [4]:
!wandb login 32d59652dce5721c31255007b9c572f3010acf97

wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Ery\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [6]:
!wandb login 32d59652dce5721c31255007b9c572f3010acf97

wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Ery\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [3]:
torch.cuda.empty_cache()

In [4]:
model = YOLO('yolo11shrimp.yaml') 

result = model.train(
    data='Shrimp-larvae-detection-1/data.yaml',
    epochs = 80,
    batch = 20,
    device=0
)

New https://pypi.org/project/ultralytics/8.3.213 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.7 torch-2.7.0+cu118 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Shrimp-larvae-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11shrimp.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, 

In [5]:
model_test = YOLO('runs/detect/train6/weights/best.pt')

In [6]:
model_test.info()

YOLO11shrimp summary: 181 layers, 4,529,453 parameters, 0 gradients, 11.0 GFLOPs


(181, 4529453, 0, 11.020726400000001)

In [23]:
import torch
import time

# ✅ Clear GPU cache before test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# ✅ Warm-up GPU (optional for consistent timing)
if device.type == "cuda":
    dummy = torch.randn(1, 3, 640, 640).to(device)
    _ = model_test(dummy)
    torch.cuda.synchronize()

# ✅ Measure inference time + GPU memory
if device.type == "cuda":
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record()
else:
    start_time = time.time()

# --- INFERENCE ---
test_pred = model_test.predict(
    source="test.jpg",
    show_labels=False,
    show_conf=False,
    save=True
)

# ✅ End timing
if device.type == "cuda":
    end_event.record()
    torch.cuda.synchronize()
    inference_time_ms = start_event.elapsed_time(end_event)
    max_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB
else:
    inference_time_ms = (time.time() - start_time) * 1000
    max_memory = 0.0

# ✅ Print metrics
print(f"⚙️ Inference Time: {inference_time_ms:.2f} ms")
print(f"💾 GPU Memory Usage: {max_memory:.2f} MB")



WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.820303916931152. Dividing input by 255.


0: 640x640 (no detections), 27.9ms
Speed: 0.0ms preprocess, 27.9ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 e:\PD1ModelTrainings\PD1ModelTrainingCodes\test.jpg: 448x640 297 shrimps, 15.4ms
Speed: 1.8ms preprocess, 15.4ms inference, 270.3ms postprocess per image at shape (1, 3, 448, 640)
Results saved to E:\PD1ModelTrainings\PD1ModelTrainingCodes\runs\detect\predict3
⚙️ Inference Time: 345.15 ms
💾 GPU Memory Usage: 85.39 MB
